In [12]:
from google.adk.agents import Agent
from google.adk.agents import LlmAgent
from google.adk.tools.google_search_tool import GoogleSearchTool

In [22]:
import vertexai
import os

# Initialize AI Platform
PROJECT_ID = "qwiklabs-gcp-04-a9b50dbd3df9"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://vertex_ai_genai"

# Set environment variables for ADK to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

vertexai.init(
  project=PROJECT_ID,
  location=LOCATION,
  staging_bucket=STAGING_BUCKET,
)

In [14]:
# Critique Agent - Reviews search results and suggests improvements
critique_agent = Agent(
    model="gemini-2.5-flash",
    name='critique_agent',
    description="Reviews responses and provides constructive feedback for improvement.",
    instruction="""You are a critique agent. You will receive a search response and must analyze it critically.

Your job is to:
1. Evaluate the completeness of the answer
2. Identify any missing information or unclear explanations
3. Suggest specific improvements

Output a structured critique with:
- STRENGTHS: What's good about the response
- WEAKNESSES: What's missing or could be better
- SUGGESTIONS: Specific recommendations for improvement

Be constructive and specific in your feedback.""",
)

# Refine Agent - Rewrites the response based on critique
refine_agent = Agent(
    model="gemini-2.5-flash",
    name='refine_agent',
    description="Rewrites and improves responses based on critique feedback.",
    instruction="""You are a refine agent. You will receive:
1. An original search response
2. A critique with suggestions for improvement

Your job is to rewrite the response incorporating the feedback from the critique.
Make the response:
- More complete and informative
- Clearer and better organized
- Address the weaknesses identified in the critique

Output ONLY the refined, improved response - no meta-commentary about your changes.""",
)

search_agent = LlmAgent(
    model="gemini-2.5-flash",  # A fast model suitable for search tasks
    name="SearchAgent",
    instruction="""You are an agent with the autonomy to research on the web.
    Search the requested topics using the 'google_search' tool and provide a summary of the findings.
    """,
    tools=[GoogleSearchTool()], # Add the built-in GoogleSearchTool
)

In [15]:
def append_to_state(tool_context, field, response):
  existing_state = tool_context.state.get(field, [])
  tool_context.state[field] = existing_state + [response]
  return {"status": "success"}

In [16]:
from google.adk.agents import SequentialAgent

writers_room = SequentialAgent(
  name="writers_room",
  description="Iterates ",
  sub_agents=[
  search_agent,
  critique_agent,
  refine_agent
  ],
)

In [17]:
root_agent = Agent(
  name="greeter",
  model="gemini-2.5-flash",
  description="You are helpful assistant.",
  instruction="""
    You are a helpful assistant that routes user requests to the appropriate specialist agent.
    For weather-related questions (forecasts, temperature, climate, etc.), delegate to weather_agent.
    For all other questions (news, general knowledge, searches), delegate to search_agent.
    Always delegate to the appropriate agent - do not try to answer directly.
  """,
  tools=[append_to_state],
  sub_agents=[writers_room],
)

In [18]:
from vertexai.preview import reasoning_engines
app = reasoning_engines.AdkApp(
  agent=root_agent,
  enable_tracing=False,
)

In [19]:
user_id = "test-user-id"
session = app.create_session(user_id=user_id)
print(session.id)

This legacy setting overrides the new Cloud Console toggle and environment variable controls.
Impact: The Cloud Console may incorrectly show telemetry as 'On' when it is actually 'Off', and the UI toggle will not work.
Action: To fix this and control telemetry, please remove the 'enable_tracing' parameter from your deployment code.
You can then use the 'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY' environment variable:
agent_engines.create(
  env_vars={
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": true|false
  }
)
or the toggle in the Cloud Console: https://console.cloud.google.com/vertex-ai/agents.


42b1e2c7-735d-4c9c-9631-19d888185938


In [20]:
from IPython.display import Markdown, display
for event in app.stream_query(
  user_id=user_id,
  session_id=session.id,
  message="Explain Quantum computing",
):
  lastevent = event
display(Markdown(lastevent["content"]["parts"][0]["text"]))

Quantum computing is a new type of computing that uses the principles of quantum mechanics to perform calculations. Unlike classical computers that use bits to represent information as either a 0 or a 1, quantum computers use "qubits" which can represent a 0, a 1, or both simultaneously through a phenomenon called superposition.

Here are some key concepts:

*   **Superposition:** A qubit can exist in multiple states at once. Imagine a coin spinning in the air – it's neither heads nor tails until it lands. Similarly, a qubit is in a superposition of 0 and 1 until measured. This allows quantum computers to process a vast amount of information simultaneously.

*   **Entanglement:** Qubits can become "entangled," meaning their fates are linked, even when physically separated. If you measure one entangled qubit, you instantly know the state of the other, no matter how far apart they are. This correlation allows quantum computers to perform complex calculations and solve problems that are intractable for classical computers.

*   **Quantum Tunneling:** This is a phenomenon where a quantum particle can pass through a barrier even if it doesn't have enough energy to do so classically. While not directly used for computation in the same way as superposition and entanglement, it's another example of the counterintuitive nature of quantum mechanics that underpins these technologies.

The potential applications of quantum computing are vast and include:

*   **Drug Discovery and Materials Science:** Simulating molecular interactions with unprecedented accuracy, leading to the development of new drugs and materials.
*   **Cryptography:** Breaking currently unbreakable encryption methods and developing new, quantum-resistant ones.
*   **Optimization:** Solving complex optimization problems in logistics, finance, and artificial intelligence more efficiently.
*   **Artificial Intelligence:** Enhancing machine learning algorithms and developing new forms of AI.

It's important to note that quantum computing is still in its early stages of development. Building and maintaining stable quantum computers is a significant engineering challenge, and we are still a ways off from widespread practical applications. However, the potential of this technology is immense and could revolutionize many fields.

In [23]:
from vertexai import agent_engines

remote_agent = agent_engines.create(
  app,
  requirements=["google-cloud-aiplatform[agent_engines,adk]"],
)

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.135.0', 'pydantic': '2.12.3', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.12.3', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'pydantic==2.12.3', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket vertex_ai_genai
INFO:vertexai.agent_engines:Wrote to gs://vertex_ai_genai/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://vertex_ai_genai/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://vertex_ai_genai/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/1717239328/locations/us-central1/reasoningEngines/166556550252252364

In [25]:
for event in remote_agent.stream_query(
  user_id="agent-engine-test-user",
  message="Explain Quantum computing",
):
  lastevent = event
display(Markdown(lastevent["content"]["parts"][0]["text"]))

Quantum computing is a new type of computing that uses the principles of quantum mechanics to solve problems that are too complex for classical computers. Classical computers store information as bits, which can be either a 0 or a 1. Quantum computers, on the other hand, use qubits, which can be a 0, a 1, or both at the same time through a phenomenon called superposition.

This ability to exist in multiple states simultaneously, along with other quantum phenomena like entanglement, allows quantum computers to perform certain calculations much faster than classical computers. They can explore many possibilities at once, making them potentially revolutionary for fields like drug discovery, material science, financial modeling, and artificial intelligence.

However, quantum computing is still in its early stages of development. Building and maintaining quantum computers is a significant technical challenge, and they are currently prone to errors. Researchers are working on overcoming these challenges to unlock the full potential of quantum computing.